# 01 - Exploratory Data Analysis: Energy Demand, Features and Business Context

This notebook is the main exploratory analysis for the **Energy Intelligence Platform**.

The goal is to understand the electricity demand dataset before modeling:

- What data do we have?
- What period does it cover?
- Are there missing values, duplicated timestamps or suspicious records?
- What are the demand patterns by hour, day, month and year?
- Are there peak periods and outliers?
- Which engineered features are useful for forecasting and risk scoring?
- How should we prepare the data for machine learning pipelines?

Business framing: this EDA supports demand forecasting, peak-risk scoring and 2030 scenario planning.

## 1. Setup

Run `python run_pipeline.py` from the project root before this notebook. The notebook reads prepared files from `data/processed/`.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

PROJECT_ROOT = Path('..').resolve()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_METRICS = PROJECT_ROOT / 'outputs' / 'metrics'
FIGSIZE = (14, 5)

print(PROJECT_ROOT)

In [ ]:
required_files = [
    DATA_PROCESSED / 'demand_clean.csv',
    DATA_PROCESSED / 'demand_features.csv',
    DATA_PROCESSED / 'annual_demand_summary.csv',
    DATA_PROCESSED / 'demand_projection_2030.csv',
    DATA_PROCESSED / 'nuclear_capacity.csv',
    DATA_PROCESSED / 'nuclear_capacity_projection_2030.csv',
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError('Run python run_pipeline.py first. Missing files: ' + ', '.join(missing_files))

## 2. Load Datasets

We load the cleaned hourly demand data, engineered features, annual summaries, 2030 demand scenarios and nuclear capacity tables.

In [ ]:
demand = pd.read_csv(DATA_PROCESSED / 'demand_clean.csv', parse_dates=['datetime'])
features = pd.read_csv(DATA_PROCESSED / 'demand_features.csv', parse_dates=['datetime'])
annual_demand = pd.read_csv(DATA_PROCESSED / 'annual_demand_summary.csv')
demand_projection = pd.read_csv(DATA_PROCESSED / 'demand_projection_2030.csv')
nuclear = pd.read_csv(DATA_PROCESSED / 'nuclear_capacity.csv')
nuclear_projection = pd.read_csv(DATA_PROCESSED / 'nuclear_capacity_projection_2030.csv')

print('Demand:', demand.shape)
print('Features:', features.shape)
print('Annual demand:', annual_demand.shape)
print('Demand projection:', demand_projection.shape)
print('Nuclear:', nuclear.shape)
print('Nuclear projection:', nuclear_projection.shape)

In [ ]:
display(demand.head())
display(features.head())
display(nuclear.head())

## 3. Dataset Overview

First, we check the time range, frequency and basic descriptive statistics. For energy demand, the timestamp structure matters as much as the numeric values.

In [ ]:
overview = {
    'start_datetime': demand['datetime'].min(),
    'end_datetime': demand['datetime'].max(),
    'rows': len(demand),
    'unique_timestamps': demand['datetime'].nunique(),
    'avg_demand_mw': demand['demand_mw'].mean(),
    'median_demand_mw': demand['demand_mw'].median(),
    'peak_demand_mw': demand['demand_mw'].max(),
    'min_demand_mw': demand['demand_mw'].min(),
}
pd.DataFrame([overview]).T.rename(columns={0: 'value'})

In [ ]:
demand['time_delta_hours'] = demand['datetime'].diff().dt.total_seconds() / 3600
demand['time_delta_hours'].value_counts(dropna=False).head(10)

Interpretation checklist:

- If most time deltas are 1 hour, the series is regular.
- Large gaps suggest missing periods or daylight-saving changes.
- Duplicate timestamps would be a serious modeling issue.

## 4. Data Quality Checks

Before modeling, we validate missing values, duplicate timestamps, impossible demand values and extreme records.

In [ ]:
quality_summary = pd.DataFrame({
    'missing_values': demand.isna().sum(),
    'missing_pct': demand.isna().mean() * 100,
    'dtype': demand.dtypes.astype(str)
})
quality_summary

In [ ]:
duplicate_timestamps = demand['datetime'].duplicated().sum()
non_positive_demand = (demand['demand_mw'] <= 0).sum()
print(f'Duplicate timestamps: {duplicate_timestamps}')
print(f'Non-positive demand values: {non_positive_demand}')

In [ ]:
demand[['demand_mw']].describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T

## 5. Demand Distribution

Distribution analysis helps identify whether the model will mostly learn normal demand or whether peak periods are rare and need special treatment.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(demand['demand_mw'], bins=60, kde=True, ax=axes[0], color='#3b82f6')
axes[0].set_title('Demand distribution')
axes[0].set_xlabel('Demand MW')
sns.boxplot(x=demand['demand_mw'], ax=axes[1], color='#93c5fd')
axes[1].set_title('Demand boxplot')
axes[1].set_xlabel('Demand MW')
plt.tight_layout()

In [ ]:
q75 = demand['demand_mw'].quantile(0.75)
q90 = demand['demand_mw'].quantile(0.90)
q95 = demand['demand_mw'].quantile(0.95)
pd.DataFrame({
    'threshold': ['75th percentile', '90th percentile', '95th percentile'],
    'demand_mw': [q75, q90, q95]
})

Business interpretation:

- The 90th percentile is a practical threshold for **high peak-risk**.
- The upper tail matters because those hours drive capacity planning, demand response and reliability risk.
- A model with low average error can still be weak if it misses high-demand hours.

## 6. Time Series View

Now we visualize the full hourly time series and resampled trends. This helps separate daily noise from seasonal and annual behavior.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(demand['datetime'], demand['demand_mw'], linewidth=0.7, color='#2563eb')
ax.set_title('Hourly demand over time')
ax.set_ylabel('Demand MW')
ax.set_xlabel('Datetime')
plt.tight_layout()

In [ ]:
demand_ts = demand.set_index('datetime')['demand_mw']
daily = demand_ts.resample('D').mean()
weekly = demand_ts.resample('W').mean()
monthly = demand_ts.resample('M').mean()

fig, ax = plt.subplots(figsize=(16, 5))
daily.plot(ax=ax, alpha=0.35, label='Daily average', color='#93c5fd')
weekly.plot(ax=ax, linewidth=1.5, label='Weekly average', color='#2563eb')
monthly.plot(ax=ax, linewidth=2.5, label='Monthly average', color='#ef4444')
ax.set_title('Demand trend at daily, weekly and monthly levels')
ax.set_ylabel('Demand MW')
ax.legend()
plt.tight_layout()

## 7. Calendar Feature Analysis

Demand is strongly affected by hour of day, day of week and month. These patterns justify the time-based features used by the forecasting model.

In [ ]:
eda = demand.copy()
eda['hour'] = eda['datetime'].dt.hour
eda['day_of_week'] = eda['datetime'].dt.day_name()
eda['day_num'] = eda['datetime'].dt.dayofweek
eda['month'] = eda['datetime'].dt.month
eda['month_name'] = eda['datetime'].dt.month_name()
eda['year'] = eda['datetime'].dt.year
eda['is_weekend'] = eda['datetime'].dt.dayofweek >= 5
eda.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.lineplot(data=eda, x='hour', y='demand_mw', estimator='mean', errorbar=None, ax=axes[0, 0], color='#2563eb')
axes[0, 0].set_title('Average demand by hour')
sns.boxplot(data=eda, x='is_weekend', y='demand_mw', ax=axes[0, 1], palette='Blues')
axes[0, 1].set_title('Demand: weekday vs weekend')
sns.lineplot(data=eda, x='month', y='demand_mw', estimator='mean', errorbar=None, ax=axes[1, 0], color='#ef4444')
axes[1, 0].set_title('Average demand by month')
sns.lineplot(data=eda, x='year', y='demand_mw', estimator='mean', errorbar=None, ax=axes[1, 1], color='#16a34a')
axes[1, 1].set_title('Average demand by year')
plt.tight_layout()

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hour_day = eda.groupby(['day_of_week', 'hour'])['demand_mw'].mean().reset_index()
heatmap_data = hour_day.pivot(index='day_of_week', columns='hour', values='demand_mw').reindex(day_order)
plt.figure(figsize=(16, 5))
sns.heatmap(heatmap_data, cmap='YlOrRd', linewidths=0.2)
plt.title('Average demand heatmap: day of week vs hour')
plt.xlabel('Hour')
plt.ylabel('Day of week')
plt.tight_layout()

Business interpretation:

- Hourly patterns help operations teams anticipate intraday peaks.
- Weekend behavior often differs from weekday behavior.
- Seasonal patterns are relevant for procurement, capacity planning and stress testing.

## 8. Peak Demand Investigation

Peak periods are not just high values. They are the operationally important tail of the distribution.

In [ ]:
eda['risk_band'] = pd.cut(
    eda['demand_mw'],
    bins=[-np.inf, q75, q90, np.inf],
    labels=['Low', 'Medium', 'High']
)
risk_summary = eda.groupby('risk_band', observed=True).agg(
    hours=('demand_mw', 'size'),
    avg_demand_mw=('demand_mw', 'mean'),
    min_demand_mw=('demand_mw', 'min'),
    max_demand_mw=('demand_mw', 'max')
).reset_index()
risk_summary['pct_hours'] = risk_summary['hours'] / len(eda) * 100
risk_summary

In [ ]:
top_peak_hours = eda.sort_values('demand_mw', ascending=False).head(20)
top_peak_hours[['datetime', 'demand_mw', 'hour', 'day_of_week', 'month_name', 'risk_band']]

In [ ]:
peak_by_month = eda[eda['risk_band'] == 'High'].groupby(['year', 'month']).size().reset_index(name='high_risk_hours')
plt.figure(figsize=(16, 5))
sns.barplot(data=peak_by_month, x='month', y='high_risk_hours', hue='year')
plt.title('High-risk hours by month and year')
plt.ylabel('High-risk hours')
plt.tight_layout()

## 9. Outlier and Anomaly Review

Outliers can be real operational peaks or data quality issues. We flag them using an IQR rule and then inspect the records.

In [ ]:
q1 = demand['demand_mw'].quantile(0.25)
q3 = demand['demand_mw'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = demand[(demand['demand_mw'] < lower_bound) | (demand['demand_mw'] > upper_bound)].copy()
print(f'IQR lower bound: {lower_bound:,.0f} MW')
print(f'IQR upper bound: {upper_bound:,.0f} MW')
print(f'Outlier rows: {len(outliers):,}')
display(outliers.sort_values('demand_mw', ascending=False).head(15))

Important: for electricity demand, high outliers may be real. We should not automatically remove peaks, because the peak-risk model is specifically designed to learn from them.

## 10. Feature Engineering Review

The model uses time, lag and rolling features. This section checks whether those features make sense statistically.

In [ ]:
features.info()

In [ ]:
feature_cols = [col for col in features.columns if col not in ['datetime', 'demand_mw']]
features[feature_cols + ['demand_mw']].describe().T

In [ ]:
corr = features[feature_cols + ['demand_mw']].corr(numeric_only=True)
plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.2)
plt.title('Feature correlation matrix')
plt.tight_layout()

In [ ]:
corr_target = corr['demand_mw'].drop('demand_mw').sort_values(key=lambda s: s.abs(), ascending=False)
corr_target.to_frame('correlation_with_demand').head(20)

Expected pattern:

- `lag_24h` and rolling means are usually strong because electricity demand has daily persistence.
- Calendar features explain recurring cycles.
- Highly correlated features are useful for tree models, but would need careful treatment in linear models.

## 11. Lag Relationship Diagnostics

Lag plots help confirm whether recent demand is predictive of current demand.

In [ ]:
lag_cols = ['lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), lag_cols):
    sample = features.sample(min(5000, len(features)), random_state=42)
    sns.scatterplot(data=sample, x=col, y='demand_mw', s=8, alpha=0.35, ax=ax)
    ax.set_title(f'Demand vs {col}')
plt.tight_layout()

## 12. Train/Test Split for Time Series

For time series, we do not randomly shuffle. We train on the earlier period and test on the later period to simulate future prediction.

In [ ]:
ordered = features.sort_values('datetime').reset_index(drop=True)
split_idx = int(len(ordered) * 0.8)
train = ordered.iloc[:split_idx].copy()
test = ordered.iloc[split_idx:].copy()

split_summary = pd.DataFrame([
    {'split': 'train', 'rows': len(train), 'start': train['datetime'].min(), 'end': train['datetime'].max()},
    {'split': 'test', 'rows': len(test), 'start': test['datetime'].min(), 'end': test['datetime'].max()},
])
split_summary

In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(train['datetime'], train['demand_mw'], label='Train', linewidth=0.7)
plt.plot(test['datetime'], test['demand_mw'], label='Test', linewidth=0.7)
plt.title('Time-based train/test split')
plt.ylabel('Demand MW')
plt.legend()
plt.tight_layout()

## 13. Sklearn Pipeline Example

This section demonstrates a clean modeling pipeline. The production script uses the project modules, but this notebook shows the mechanics for interview explanation.

In [ ]:
numeric_features = [
    'hour', 'day_of_week', 'month', 'year', 'day_of_year', 'is_weekend', 'is_business_hour',
    'lag_1h', 'lag_2h', 'lag_24h', 'lag_48h', 'lag_168h',
    'rolling_mean_6h', 'rolling_mean_24h', 'rolling_mean_168h'
]

X_train = train[numeric_features]
y_train = train['demand_mw']
X_test = test[numeric_features]
y_test = test['demand_mw']

preprocess = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features)
    ]
)

rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', RandomForestRegressor(n_estimators=80, max_depth=12, random_state=42, n_jobs=-1))
])

rf_pipeline

In [ ]:
rf_pipeline.fit(X_train, y_train)
pred = rf_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
mape = np.mean(np.abs((y_test - pred) / y_test)) * 100

pd.DataFrame([{'MAE': mae, 'RMSE': rmse, 'MAPE_pct': mape}])

In [ ]:
model_results = test[['datetime', 'demand_mw']].copy()
model_results['prediction_mw'] = pred
model_results['absolute_error_mw'] = (model_results['demand_mw'] - model_results['prediction_mw']).abs()

plt.figure(figsize=(16, 5))
plot_sample = model_results.tail(24 * 30)
plt.plot(plot_sample['datetime'], plot_sample['demand_mw'], label='Actual', linewidth=1)
plt.plot(plot_sample['datetime'], plot_sample['prediction_mw'], label='Prediction', linewidth=1)
plt.title('Random Forest pipeline: last 30 days of test predictions')
plt.ylabel('Demand MW')
plt.legend()
plt.tight_layout()

Pipeline interpretation:

- The `ColumnTransformer` makes preprocessing explicit.
- `SimpleImputer` protects the model if future data has missing features.
- `StandardScaler` is not required for Random Forest, but shows how a reusable preprocessing stage would work for linear models.
- The train/test split respects time order.

## 14. Error Analysis

A good EDA does not stop at metrics. We inspect when and where the model makes mistakes.

In [ ]:
model_results['hour'] = model_results['datetime'].dt.hour
model_results['month'] = model_results['datetime'].dt.month
model_results['day_of_week'] = model_results['datetime'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=model_results, x='hour', y='absolute_error_mw', ax=axes[0], color='#93c5fd')
axes[0].set_title('Absolute error by hour')
sns.boxplot(data=model_results, x='month', y='absolute_error_mw', ax=axes[1], color='#bfdbfe')
axes[1].set_title('Absolute error by month')
plt.tight_layout()

In [ ]:
model_results.sort_values('absolute_error_mw', ascending=False).head(20)

Business interpretation:

- If errors concentrate in summer or winter, weather features should be added.
- If errors concentrate in specific hours, intraday behavior needs better features.
- If peak errors are large, the model may still be risky for capacity planning even with acceptable average metrics.

## 15. Feature Importance

Feature importance helps explain the model to stakeholders. For tree models, lag and rolling features usually dominate.

In [ ]:
feature_importance = pd.DataFrame({
    'feature': numeric_features,
    'importance': rf_pipeline.named_steps['model'].feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature', color='#2563eb')
plt.title('Top feature importances')
plt.tight_layout()
feature_importance.head(15)

## 16. 2030 Demand Scenario EDA

The long-range 2030 view is not a precise hourly forecast. It is a scenario model designed for strategic discussion.

The scenario assumptions include:

- structural demand growth,
- AI / data center load,
- electrification,
- blockchain / crypto load,
- efficiency offsets.

In [ ]:
display(demand_projection.head())
display(demand_projection[demand_projection['year'] == 2030][[
    'scenario', 'projected_avg_demand_mw', 'projected_peak_demand_mw',
    'net_avg_growth_pct', 'net_peak_growth_pct'
]].sort_values('projected_peak_demand_mw'))

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(data=demand_projection, x='year', y='projected_peak_demand_mw', hue='scenario', marker='o')
plt.title('Projected peak demand scenarios to 2030')
plt.ylabel('Projected peak demand MW')
plt.tight_layout()

How to explain this:

- The historical model predicts short-term hourly demand.
- The 2030 model explores strategic scenarios.
- AI/data centers and blockchain are modeled as additional load drivers.
- Efficiency acts as an offset.
- The output is useful for planning conversations, not real-time dispatch.

## 17. Nuclear Capacity EDA

The nuclear module adds the supply-side planning context: how much stable baseload capacity may be available by region and country.

In [ ]:
display(nuclear)
nuclear.groupby('status').agg(
    reactors=('reactor_count', 'sum'),
    capacity_mw=('capacity_mw', 'sum'),
    annual_generation_gwh=('annual_generation_gwh', 'sum')
).sort_values('capacity_mw', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
country_capacity = nuclear.groupby('country', as_index=False)['capacity_mw'].sum().sort_values('capacity_mw', ascending=False).head(12)
region_capacity = nuclear.groupby('region', as_index=False)['capacity_mw'].sum().sort_values('capacity_mw', ascending=False)
sns.barplot(data=country_capacity, x='capacity_mw', y='country', ax=axes[0], color='#2563eb')
axes[0].set_title('Nuclear capacity by country')
sns.barplot(data=region_capacity, x='capacity_mw', y='region', ax=axes[1], color='#16a34a')
axes[1].set_title('Nuclear capacity by region')
plt.tight_layout()

In [ ]:
nuclear_2030 = nuclear_projection.groupby(['year', 'scenario'], as_index=False)['projected_capacity_mw'].sum()
plt.figure(figsize=(14, 5))
sns.lineplot(data=nuclear_2030, x='year', y='projected_capacity_mw', hue='scenario', marker='o')
plt.title('Nuclear capacity scenarios to 2030')
plt.ylabel('Projected capacity MW')
plt.tight_layout()
display(nuclear_2030[nuclear_2030['year'] == 2030])

## 18. Combined Demand and Nuclear Planning Question

A useful executive question is:

> If peak demand rises by 2030, how much stable capacity could nuclear provide in the broader strategic supply mix?

This project does not claim nuclear capacity alone should cover demand. The point is to connect demand pressure with baseload capacity planning.

In [ ]:
base_demand_2030 = demand_projection[(demand_projection['scenario'] == 'Base') & (demand_projection['year'] == 2030)].iloc[0]
base_nuclear_2030 = nuclear_2030[(nuclear_2030['scenario'] == 'Base') & (nuclear_2030['year'] == 2030)].iloc[0]

planning_summary = pd.DataFrame([
    {'metric': 'Projected peak demand 2030 - Base', 'mw': base_demand_2030['projected_peak_demand_mw']},
    {'metric': 'Projected average demand 2030 - Base', 'mw': base_demand_2030['projected_avg_demand_mw']},
    {'metric': 'Projected nuclear capacity 2030 - Base sample', 'mw': base_nuclear_2030['projected_capacity_mw']},
])
planning_summary

## 19. Key EDA Findings

Use this section as interview notes.

1. The demand series is hourly and suitable for time-series feature engineering.
2. Demand has strong intraday, weekly and seasonal patterns.
3. Peak periods are rare but operationally important, which motivates risk scoring.
4. Lag and rolling features are strongly related to demand.
5. A time-based split is required to avoid leakage.
6. The 2030 view should be described as scenario planning, not exact forecasting.
7. Nuclear capacity analytics provides strategic baseload context, not a direct one-to-one solution for local PJM demand.

Recommended next data additions:

- weather by region,
- holidays and calendar events,
- electricity prices,
- outage data,
- data center load forecasts,
- EV adoption and electrification scenarios,
- updated official IAEA PRIS/RDS-1 exports.